In [1]:
import os
import oracledb
from dotenv import load_dotenv
import csv
from datetime import datetime

load_dotenv()

connection = oracledb.connect(
    user=os.getenv("ORACLE_USER"),
    password=os.getenv("ORACLE_PASSWORD"),
    dsn=oracledb.makedsn("localhost", 1522, service_name="stu"),
)
cursor = connection.cursor()
print("Connected to Oracle")

Connected to Oracle


# setup tables

In [2]:
CHANNELS_TABLE = "youtube_channels"
VIDEOS_TABLE = "youtube_videos"
COMMENTS_TABLE = "youtube_comments"
TABLES = [COMMENTS_TABLE, VIDEOS_TABLE, CHANNELS_TABLE]


def setup_tables():
    for table in TABLES:
        try:
            cursor.execute(f"DROP TABLE {table}")
            print(f"Dropped {table}")
        except oracledb.DatabaseError as e:
            print(f"{table} did not exist: {e}")
    cursor.execute("PURGE RECYCLEBIN")
    print("Recycle bin purged")

    cursor.execute(
        f"""
        CREATE TABLE {CHANNELS_TABLE} (
            channel_id        VARCHAR2(50) PRIMARY KEY,
            channel_name      VARCHAR2(100) NOT NULL,
            lean              VARCHAR2(20)  NOT NULL,
            description       VARCHAR2(500),
            published_at      DATE,
            subscriber_count  NUMBER,
            total_view_count  NUMBER,
            total_video_count NUMBER
        )
    """
    )
    print("Created youtube_channels table")

    cursor.execute(
        f"""
        CREATE TABLE {VIDEOS_TABLE} (
            video_id      VARCHAR2(50) PRIMARY KEY,
            channel_id    VARCHAR2(50) NOT NULL,
            title         VARCHAR2(200),
            lean          VARCHAR2(20) NOT NULL,
            published_at  DATE,
            year          NUMBER(4),
            month         NUMBER(2),
            view_count    NUMBER,
            like_count    NUMBER,
            comment_count NUMBER,
            CONSTRAINT fk_video_channel FOREIGN KEY (channel_id)
                REFERENCES {CHANNELS_TABLE}(channel_id)
        )
    """
    )
    print("Created youtube_videos table")

    cursor.execute(
        f"""
        CREATE TABLE {COMMENTS_TABLE} (
            comment_id   VARCHAR2(100) PRIMARY KEY,
            video_id     VARCHAR2(50) NOT NULL,
            text         VARCHAR2(500),
            like_count   NUMBER,
            published_at DATE,
            sentiment_score NUMBER,
            sentiment_label VARCHAR2(20),
            CONSTRAINT fk_comment_video FOREIGN KEY (video_id)
                REFERENCES {VIDEOS_TABLE}(video_id)
        )
    """
    )
    print("Created youtube_comments table")

# insert from csv to sql

In [3]:
def parse_date(s):
    if not s:
        return None
    return datetime.strptime(s[:19], "%Y-%m-%dT%H:%M:%S")

In [15]:
CLEANED_DATA_PREFIX = "../../cleaned_data/youtube/"


def insert_channels():
    rows = []
    with open(
        os.path.join(CLEANED_DATA_PREFIX, "channels.csv"), newline="", encoding="utf-8"
    ) as f:
        for r in csv.DictReader(f):
            rows.append(
                (
                    r["channel_id"],
                    r["channel_name"],
                    r["lean"],
                    r["description"][:495],
                    parse_date(r["published_at"]),
                    int(r["subscriber_count"]),
                    int(r["total_view_count"]),
                    int(r["total_video_count"]),
                )
            )
    cursor.executemany(
        f"""
        INSERT INTO {CHANNELS_TABLE}
        (channel_id, channel_name, lean, description, published_at,
         subscriber_count, total_view_count, total_video_count)
        VALUES (:1,:2,:3,:4,:5,:6,:7,:8)
    """,
        rows,
    )
    print(f"Inserted {len(rows)} channels into {CHANNELS_TABLE}")


def insert_videos():
    rows = []
    with open(
        os.path.join(CLEANED_DATA_PREFIX, "top_5_videos.csv"),
        newline="",
        encoding="utf-8",
    ) as f:
        for r in csv.DictReader(f):
            rows.append(
                (
                    r["video_id"],
                    r["channel_id"],
                    r["title"][:200],
                    r["lean"],
                    parse_date(r["published_at"]),
                    int(r["year"]),
                    int(r["month"]) ,
                    int(r["view_count"]),
                    int(r["like_count"]),
                    int(r["comment_count"]),
                )
            )
    cursor.executemany(
        f"""
        INSERT INTO {VIDEOS_TABLE}
        (video_id, channel_id, title, lean, published_at,
         year, month, view_count, like_count, comment_count)
        VALUES (:1,:2,:3,:4,:5,:6,:7,:8,:9,:10)
    """,
        rows,
    )
    print(f"Inserted {len(rows)} videos into {VIDEOS_TABLE}")


def insert_comments():
    rows = []
    with open(
        os.path.join(CLEANED_DATA_PREFIX, "comments_with_sentiment.csv"), newline="", encoding="utf-8"
    ) as f:
        for r in csv.DictReader(f):
            rows.append(
                (
                    r["comment_id"],
                    r["video_id"],
                    r["text"][:485],
                    int(r["like_count"]) if r["like_count"] else 0,
                    parse_date(r["published_at"]),
                    float(r["sentiment_score"]),
                    r["sentiment_label"],
                )
            )
    cursor.executemany(
        f"""
        INSERT INTO {COMMENTS_TABLE}
        (comment_id, video_id, text, like_count, published_at, sentiment_score, sentiment_label)
        VALUES (:1,:2,:3,:4,:5,:6,:7)
    """,
        rows,
    )
    print(f"Inserted {len(rows)} comments into {COMMENTS_TABLE}")

In [16]:
setup_tables()
insert_channels()
insert_videos()
insert_comments()
connection.commit()

Dropped youtube_comments
Dropped youtube_videos
Dropped youtube_channels
Recycle bin purged
Created youtube_channels table
Created youtube_videos table
Created youtube_comments table
Inserted 2 channels into youtube_channels
Inserted 960 videos into youtube_videos
Inserted 1240 comments into youtube_comments


# look at data

In [17]:
import pandas as pd

query = f"""
SELECT *
FROM {CHANNELS_TABLE} c
"""
cursor.execute(query)

rows = cursor.fetchall()
df = pd.DataFrame(rows, columns=[d[0] for d in cursor.description])
df

,CHANNEL_ID,CHANNEL_NAME,LEAN,DESCRIPTION,PUBLISHED_AT,SUBSCRIBER_COUNT,TOTAL_VIEW_COUNT,TOTAL_VIDEO_COUNT
0,UCnQC_G5Xsjhp9fEJKuIcrSw,Ben Shapiro,conservative,Ben Shapiro is a renowned conservative politic...,2016-11-01 00:13:29,7090000,4649859370,9732
1,UC1yBKRuGpC1tSM73A0ZjYjQ,The Young Turks,liberal,The Young Turks is The Online News Show. Join ...,2005-12-21 20:46:51,6500000,7311682837,71052


In [ ]:
import pandas as pd

query = f"""
SELECT *
FROM {VIDEOS_TABLE} c
"""
cursor.execute(query)

rows = cursor.fetchall()
df = pd.DataFrame(rows, columns=[d[0] for d in cursor.description])
df

,VIDEO_ID,CHANNEL_ID,TITLE,LEAN,PUBLISHED_AT,YEAR,MONTH,VIEW_COUNT,LIKE_COUNT,COMMENT_COUNT
0,VuEHQwgHKjI,UCnQC_G5Xsjhp9fEJKuIcrSw,My Response to Steven Crowder's Despicable Bet...,conservative,2023-01-20 20:47:35,2023,1,2035210,72031,29878
1,R4cIapZx5Vg,UCnQC_G5Xsjhp9fEJKuIcrSw,Ben Reacts To a Void,conservative,2023-01-17 23:00:10,2023,1,1020335,40731,8172
2,O5By03X4--8,UCnQC_G5Xsjhp9fEJKuIcrSw,Andrew Tate Predicted His Arrest,conservative,2023-01-04 20:23:05,2023,1,2490134,95696,5641
3,GM7Qzk_8m1Q,UCnQC_G5Xsjhp9fEJKuIcrSw,Here Is What I Actually Think Of Andrew Tate,conservative,2023-01-03 23:30:01,2023,1,3051598,93014,14046
4,DG2YH3EiJSs,UCnQC_G5Xsjhp9fEJKuIcrSw,Ben Reacts to Taylor Swift's Documentary,conservative,2022-12-30 22:15:00,2022,12,4779376,182121,7237
...,...,...,...,...,...,...,...,...,...,...
955,IWCzniTvdyc,UC1yBKRuGpC1tSM73A0ZjYjQ,"The Young Turks 01.29.18: Andrew McCabe, Trump...",liberal,2018-01-30 03:24:00,2018,1,61,1,0
956,f8KYa5n4IEw,UC1yBKRuGpC1tSM73A0ZjYjQ,"The Young Turks 01.19.18: Opioid Crisis, Carl ...",liberal,2018-01-20 02:41:00,2018,1,26,0,0
957,8hJJfamTeJo,UC1yBKRuGpC1tSM73A0ZjYjQ,"The Young Turks 01.08.18: Stephen Miller, Jake...",liberal,2018-01-09 02:37:00,2018,1,158,0,1
958,xCf9yKmV7hA,UC1yBKRuGpC1tSM73A0ZjYjQ,"The Young Turks 01.04.18: Trump Is A Child, Ba...",liberal,2018-01-05 20:21:00,2018,1,30,1,0


In [19]:
import pandas as pd

query = f"""
SELECT *
FROM {COMMENTS_TABLE} c
"""
cursor.execute(query)

rows = cursor.fetchall()
df = pd.DataFrame(rows, columns=[d[0] for d in cursor.description])
df

,COMMENT_ID,VIDEO_ID,TEXT,LIKE_COUNT,PUBLISHED_AT,SENTIMENT_SCORE,SENTIMENT_LABEL
0,Ugz-c8bp37Wyvq5yakN4AaABAg,sYZ19Lh_iXA,The fact she thinks Trump is struggling to go ...,3763,2024-08-17 17:16:25,-0.3818,negative
1,UgwBLTIittp-F7KzFRJ4AaABAg,sYZ19Lh_iXA,That's the best joke Colbert ever told.,2979,2024-08-17 17:14:22,0.7506,positive
2,UgxwpHEsMU-GFtzGdrR4AaABAg,sYZ19Lh_iXA,"The audience isn't fooled, but they have chose...",2819,2024-08-17 17:09:29,0.0752,positive
3,Ugx5mYhH7tlVyGuARhx4AaABAg,4is_L8jgNTo,"That's cool. It's like God is saying, ""No, you...",5417,2024-08-12 11:55:05,0.8462,positive
4,Ugz7Es97_Ulh6tknGQF4AaABAg,4is_L8jgNTo,So humbling. The root of bitterness is unforgi...,1840,2024-08-12 20:35:44,-0.4019,negative
...,...,...,...,...,...,...,...
1235,UgwEqS5mnrd1Iu3rMKV4AaABAg,F2ISI3Mq-RI,"To be honest, Trump took it easy on Biden last...",342,2024-06-29 04:51:08,0.7351,positive
1236,Ugx5zZuj1EJrx565OgJ4AaABAg,F2ISI3Mq-RI,Trump actually showed restraint on bashing Bid...,148,2024-06-29 03:08:06,0.0000,neutral
1237,Ugx6ecaov0ZjL_LhC4t4AaABAg,F2ISI3Mq-RI,You could tell that even Trump felt sorry for ...,146,2024-06-29 21:49:27,-0.0772,negative
1238,UgxIxeU-LmiHSYJaeRN4AaABAg,pzRdnNYuLlM,Face off? Ok\nThis video is 12days old\n253 vi...,1,2024-01-23 01:06:07,0.6808,positive


In [18]:
import pandas as pd

query = f"""
SELECT c.channel_name, v.title as video_title, v.published_at, v.view_count, v.like_count, v.comment_count
FROM {CHANNELS_TABLE} c
JOIN {VIDEOS_TABLE} v ON c.channel_id = v.channel_id
ORDER BY v.view_count DESC
"""
cursor.execute(query)

rows = cursor.fetchall()
df = pd.DataFrame(rows, columns=[d[0] for d in cursor.description])
df

,CHANNEL_NAME,VIDEO_TITLE,PUBLISHED_AT,VIEW_COUNT,LIKE_COUNT,COMMENT_COUNT
0,Ben Shapiro,Equality and Opportunity,2023-08-26 19:00:32,49165283,2095594,32442
1,Ben Shapiro,Judge Dismisses Veteran's Case,2024-03-05 02:00:22,30946063,1581251,25911
2,Ben Shapiro,Raven-Symoné: “I’m an American”,2023-12-25 20:00:22,28052279,1507684,77213
3,Ben Shapiro,Exposing Jordan Peterson,2023-09-19 19:00:14,27606881,1211906,16951
4,Ben Shapiro,Marriage is the answer,2024-09-22 23:00:12,23614518,628418,8241
...,...,...,...,...,...,...
955,The Young Turks,BoogieDown,2020-05-08 23:35:52,8,0,0
956,The Young Turks,Part 1: Flip Georgia Get Out the Vote Special,2020-12-29 08:00:00,7,0,0
957,The Young Turks,Another One,2020-05-14 21:53:55,7,0,0
958,The Young Turks,"The Young Turks - December 21, 2020",2020-12-22 02:37:59,7,0,0


In [ ]:
cursor.close()
connection.close()

InterfaceError: DPY-1006: cursor is not open